In [13]:
import pandas as pd
import matplotlib.pyplot as plt
from features import (feature_total_pages_count,
                      total_number_of_sessions,
                      time_last_used,
                      generate_time_features,
                      last_level_of_each_customer,
                      device_used_by_user)

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

In [52]:
df_test = pd.read_parquet("../../data/churn-prediction-25-26/test.parquet")
page_features_df = feature_total_pages_count(df_test)
tnos_features_df = total_number_of_sessions(df = df_test)
time_last_used_df = time_last_used(df = df_test)
generate_time_features_df = generate_time_features(df = df_test)
last_level_of_each_customer_df = last_level_of_each_customer(df = df_test)
device_used_by_user_df = device_used_by_user(df = df_test)

page_features_df.reset_index(inplace=True)
df_test = page_features_df.merge(tnos_features_df, how = "inner", on = "userId")

#df_test["mean_tbs_minutes"] = df_test["mean_tbs_minutes"].fillna(1000)
#df_test["median_tbs_minutes"] = df_test["median_tbs_minutes"].fillna(1000)

In [53]:
df_test = df_test.merge(time_last_used_df, on = "userId")

In [54]:
df_test = df_test.merge(generate_time_features_df, on = "userId")

In [55]:
df_test = df_test.merge(device_used_by_user_df, on = "userId")

In [56]:
df_test["mean_tbs_minutes"] = df_test["mean_tbs_minutes"].fillna(1000)
df_test["median_tbs_minutes"] = df_test["median_tbs_minutes"].fillna(1000)

In [57]:
df_test

,userId,About,Add Friend,Add to Playlist,Downgrade,Error,Help,Home,Login,Logout,...,Thumbs Down,Thumbs Up,Upgrade,total_number_of_sessions,time_not_used,mean_tps_minutes,mean_tbs_minutes,median_tps_minutes,median_tbs_minutes,exact_device
0,1000655,1.0,4.0,8.0,0.0,0.0,0.0,18.0,0.0,10.0,...,2.0,14.0,3.0,10,6105.0,107.0,6931.0,65.0,4285.0,Macintosh
1,1000963,3.0,48.0,61.0,19.0,3.0,13.0,81.0,0.0,18.0,...,20.0,120.0,6.0,21,2871.0,417.0,3125.0,316.0,2953.0,Windows
2,1001129,2.0,12.0,20.0,6.0,3.0,5.0,26.0,0.0,5.0,...,10.0,25.0,0.0,8,1503.0,285.0,9852.0,165.0,3756.0,Macintosh
3,1001963,1.0,16.0,21.0,0.0,1.0,6.0,33.0,0.0,12.0,...,10.0,26.0,3.0,16,200.0,139.0,3858.0,53.0,3458.0,Macintosh
4,1002283,8.0,68.0,91.0,40.0,2.0,27.0,136.0,0.0,46.0,...,36.0,172.0,4.0,24,342.0,551.0,3049.0,235.0,1915.0,Macintosh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2899,1999455,4.0,20.0,29.0,14.0,3.0,10.0,51.0,0.0,17.0,...,50.0,66.0,0.0,18,4697.0,323.0,3390.0,189.0,2720.0,iPhone
2900,1999691,2.0,21.0,28.0,10.0,3.0,11.0,47.0,0.0,17.0,...,11.0,67.0,2.0,14,4202.0,343.0,4325.0,308.0,2607.0,Macintosh
2901,1999720,11.0,77.0,103.0,36.0,4.0,16.0,112.0,0.0,32.0,...,28.0,182.0,1.0,16,6566.0,866.0,3890.0,605.0,2712.0,Windows
2902,1999908,3.0,26.0,43.0,6.0,2.0,6.0,72.0,0.0,29.0,...,37.0,60.0,5.0,38,2408.0,128.0,1864.0,58.0,1266.0,Windows


In [58]:
df_train = pd.read_parquet("../../data/churn-prediction-25-26/train.parquet")
page_features_df = feature_total_pages_count(df_train)
tnos_features_df = total_number_of_sessions(df = df_train)
time_last_used_df = time_last_used(df = df_train)
generate_time_features_df = generate_time_features(df = df_train)
last_level_of_each_customer_df = last_level_of_each_customer(df = df_train)
device_used_by_user_df = device_used_by_user(df = df_train)

page_features_df.reset_index(inplace=True)
df_train = page_features_df.merge(tnos_features_df, how = "inner", on = "userId")
df_train = df_train.merge(time_last_used_df, on = "userId")
df_train = df_train.merge(generate_time_features_df, on = "userId")
#df_train = df_train.merge(last_level_of_each_customer_df, on = "userId")
df_train = df_train.merge(device_used_by_user_df, on = "userId")

df_train["mean_tbs_minutes"] = df_train["mean_tbs_minutes"].fillna(1000)
df_train["median_tbs_minutes"] = df_train["median_tbs_minutes"].fillna(1000)

In [59]:
df_train

,userId,About,Add Friend,Add to Playlist,Cancel,Cancellation Confirmation,Downgrade,Error,Help,Home,...,Thumbs Down,Thumbs Up,Upgrade,total_number_of_sessions,time_not_used,mean_tps_minutes,mean_tbs_minutes,median_tps_minutes,median_tbs_minutes,exact_device
0,1000025,1.0,30.0,53.0,1.0,1.0,18.0,1.0,8.0,77.0,...,13.0,94.0,1.0,17,46286.0,404.0,1457.0,341.0,1078.0,Windows
1,1000035,2.0,23.0,27.0,0.0,0.0,7.0,1.0,5.0,54.0,...,15.0,117.0,5.0,21,6966.0,243.0,2886.0,198.0,2179.0,Linux
2,1000083,0.0,6.0,8.0,1.0,1.0,2.0,0.0,2.0,21.0,...,2.0,21.0,3.0,11,55555.0,186.0,1557.0,121.0,784.0,Windows
3,1000103,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,6.0,...,1.0,2.0,1.0,3,16171.0,75.0,25222.0,49.0,18592.0,Linux
4,1000164,2.0,17.0,24.0,0.0,0.0,10.0,1.0,4.0,40.0,...,6.0,38.0,1.0,15,655.0,230.0,4969.0,142.0,2774.0,Windows
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19135,1999781,6.0,51.0,86.0,0.0,0.0,19.0,5.0,19.0,120.0,...,30.0,152.0,4.0,37,438.0,357.0,1972.0,146.0,1435.0,Macintosh
19136,1999847,2.0,8.0,8.0,1.0,1.0,8.0,0.0,1.0,24.0,...,6.0,9.0,4.0,4,47389.0,336.0,6122.0,183.0,3931.0,Windows
19137,1999848,1.0,34.0,26.0,0.0,0.0,3.0,4.0,12.0,68.0,...,12.0,57.0,14.0,29,2491.0,149.0,2441.0,98.0,1908.0,Windows
19138,1999892,0.0,9.0,11.0,1.0,1.0,0.0,0.0,2.0,27.0,...,11.0,13.0,4.0,11,34653.0,120.0,3042.0,123.0,1699.0,Windows


In [63]:
columns_X = ['userId', 'About', 'Add Friend', 'Add to Playlist', 'Downgrade',
             'Error', 'Help', 'Home', 'Logout', 'NextSong', 'Roll Advert',
             'Save Settings', 'Settings',
       'Submit Downgrade', 'Submit Upgrade', 'Thumbs Down', 'Thumbs Up',
       'Upgrade', 'total_number_of_sessions', 'time_not_used',
       'mean_tps_minutes', 'mean_tbs_minutes', 'median_tps_minutes',
       'median_tbs_minutes', 'exact_device']


columns_y = ['Cancellation Confirmation']

In [64]:
# 4) Full pipeline
 # 3) Model
# 2) Preprocess: impute + one-hot for categoricals, impute for numerics
numerical_columns = ['total_number_of_sessions', 'time_not_used',
       'mean_tps_minutes', 'mean_tbs_minutes', 'median_tps_minutes',
       'median_tbs_minutes']

categorical_columns = ['exact_device']


categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))]
    )

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])


xgb_tuned = XGBClassifier(
    n_estimators=1362,
    learning_rate=0.06136209702804419,
    max_depth=2,
    random_state=42,
    n_jobs=-1,
)


preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline, categorical_columns),
        ("num", numeric_pipeline, numerical_columns),
    ],
    remainder="passthrough"
)

clf_tuned = Pipeline(steps=[
                        ("preprocess", preprocess),
                        ("model", xgb_tuned)
])

In [65]:
clf_tuned.fit(df_train[columns_X], df_train[columns_y])

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [66]:
pred = clf_tuned.predict(df_test[columns_X])

In [67]:
pred

array([0, 0, 0, ..., 0, 0, 0])

In [68]:
submission_df = pd.concat((df_test["userId"], pd.DataFrame(pred, columns=["target"])), axis = 1)

In [69]:
submission_df

,userId,target
0,1000655,0
1,1000963,0
2,1001129,0
3,1001963,0
4,1002283,0
...,...,...
2899,1999455,0
2900,1999691,0
2901,1999720,0
2902,1999908,0


In [70]:
submission_df.to_csv("Test1.csv", index=False)


In [71]:
len(submission_df)

2904